# data split
by DM

In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/dongmin/robot-radio-station


In [2]:
from pathlib import Path
from collections import defaultdict, Counter

import math
import random

import json
import csv

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import IPython.display as ipd

import torch

from gensim.models import Word2Vec
from rrs import cluster_utils

clean_uri = lambda uri: uri.split(':')[-1]

/home/dongmin/miniconda3/envs/rrs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
HOME = Path.home()
CWD = Path.cwd()

In [4]:
DATASET_DIR = HOME / 'userdata' / 'dongmin' / 'smp_dataset'
DATA_DIR = DATASET_DIR / 'data'
META_DIR = CWD / 'metadata'
CRAWL_DIR = CWD / 'data'

## Plain Split 

In [5]:
playlists = torch.load(CRAWL_DIR / 'playlists_filtered.pt')

/tmp/ipykernel_1172918/1727047960.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  playlists = torch.load(CRAWL_DIR / 'playlists_filtered.pt')


In [6]:
random.seed(42)
playlists = random.sample(playlists, len(playlists)) # shuffle

train_len = int(len(playlists) * 0.9)

train_playlists = playlists[:train_len]
valid_playlists = playlists[train_len:]

In [7]:
torch.save(train_playlists, CWD / 'metadata' / 'train-segments.pt')
torch.save(valid_playlists, CWD / 'metadata' / 'valid-segments.pt')

In [8]:
torch.save(playlists[:3000], CWD / 'metadata' / 'cluster_debug' / 'train-segments.pt')
torch.save(playlists[3000:6000], CWD / 'metadata' / 'cluster_debug' / 'valid-segments.pt')

## Reduced Split

### Filter playlist w/o track with audio features

In [ ]:
playlists = torch.load(CRAWL_DIR / 'playlists_filtered.pt')

In [6]:
with open(CWD / 'data' / 'total_audio_features_250328_drop.csv', 'r') as f:
  reader = csv.reader(f)
  header = next(reader)
  track_uri_w_features = [ t_uri for t_uri, *_ in reader ]

In [7]:
def filter_playlists_with_audio_features(playlists, track_uri_w_features):
  feature_tracks_set = set(track_uri_w_features)
  
  filtered_playlists = []
  
  for playlist in playlists:
    track_w_features_included = any(
      t['track_uri'] in feature_tracks_set
      for t in playlist['tracks']
    )
    if track_w_features_included:
      filtered_playlists.append(playlist)
  
  return filtered_playlists


filtered_playlists = filter_playlists_with_audio_features(
  playlists,
  track_uri_w_features
)

In [8]:
len(filtered_playlists)

160123

In [9]:
filtered_playlists[0]

{'name': 'mat',
 'collaborative': 'false',
 'pid': 3,
 'modified_at': 1501027200,
 'num_tracks': 126,
 'num_albums': 107,
 'num_followers': 1,
 'tracks': [{'pos': 0,
   'artist_name': 'Camille Saint-Saëns',
   'track_uri': 'spotify:track:4WJ7UMD4i6DOPzyXU5pZSz',
   'artist_uri': 'spotify:artist:436sYg6CZhNefQJogaXeK0',
   'track_name': 'Danse macabre',
   'album_uri': 'spotify:album:0T9YCy8TruLD6Z4qiCGSn6',
   'duration_ms': 428560,
   'album_name': 'French Festival'},
  {'pos': 2,
   'artist_name': 'No Vacation',
   'track_uri': 'spotify:track:6lbLn5iL2NBJnbib7bTXMn',
   'artist_uri': 'spotify:artist:32zeX1IoVKAGWMyy1isKUq',
   'track_name': 'Dræm Girl',
   'album_uri': 'spotify:album:3Ahz3KpKarLfZAb2bYEe65',
   'duration_ms': 234606,
   'album_name': 'Summer Break Mixtape'},
  {'pos': 3,
   'artist_name': 'No Vacation',
   'track_uri': 'spotify:track:1rEsbyBuQ0MqKF3dG7B0bf',
   'artist_uri': 'spotify:artist:32zeX1IoVKAGWMyy1isKUq',
   'track_name': 'Sad Valentine',
   'album_uri': 's

In [10]:
torch.save(filtered_playlists, CWD / 'metadata' / 'playlists_reduced.pt')

### Split reduced data

In [5]:
filtered_playlists = torch.load(CWD / 'metadata' / 'playlists_reduced.pt')

/tmp/ipykernel_1180393/3270158307.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  filtered_playlists = torch.load(CWD / 'metadata' / 'playlists_reduced.pt')


In [6]:
random.seed(42)
filtered_playlists = random.sample(
  filtered_playlists, 
  len(filtered_playlists)
)

In [7]:
num_train = 50_000
num_valid = 12_500

In [8]:
train_data = filtered_playlists[:num_train]
valid_data = filtered_playlists[num_train:num_train+num_valid]

In [9]:
torch.save(train_data, CWD / 'metadata' / 'cluster_reduced' / 'train-segments.pt')
torch.save(valid_data, CWD / 'metadata' / 'cluster_reduced' / 'valid-segments.pt')